# 🧺 **Filter Methods for Feature Selection**

Filter methods are model-independent — they evaluate the intrinsic properties of features (like correlation or statistical relevance) without involving any machine learning models.

In [19]:
from sklearn.datasets import load_iris

data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names

---
---
# **📈 Correlation-Based Methods**

| Method                   | Target Type | Feature Type | Description                                                               |
| ------------------------ | ----------- | ------------ | ------------------------------------------------------------------------- |
| **✅ Pearson Correlation**  | Continuous  | Continuous   | Measures linear relationship between feature and target                   |
| **✅ Spearman Correlation** | Continuous  | Continuous   | Measures monotonic relationship (rank-based) — good for non-linear trends |
| **Kendall Tau**          | Continuous  | Continuous   | Measures ordinal association — more robust for small samples              |
| **✅ Cramér's V**           | Categorical | Categorical  | Measures association strength between two categorical variables           |
| **Point-Biserial**       | Binary      | Continuous   | Specialized version of Pearson for binary targets                         |
| **Phi Coefficient**      | Binary      | Binary       | Equivalent to Pearson for 2 binary vars                                   |


---
## 📈 **Pearson Correlation**
---

Pearson correlation measures the **strength and direction** of a **linear relationship** between a numerical feature and a continuous target.  

Value ranges from:  
- `+1`: Perfect positive correlation  
- `-1`: Perfect negative correlation  
- `0`: No linear correlation  

| ✅ Use When                                      | ❌ Avoid When                          |
|--------------------------------------------------|----------------------------------------|
| Features are **numerical**                      | Features or target are **categorical** |
| Target is **continuous** (e.g., regression)     | Relationship is **non-linear**         |
| Need a **quick, interpretable filter method**   | Data contains **outliers**             |
| Looking for **linear relationships**            |                                        |

In [24]:
import pandas as pd
from sklearn.datasets import load_diabetes

# Load dataset
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# Compute Pearson correlation between each feature and the target
correlations = X.corrwith(y)

# Create a DataFrame for better readability
corr_df = pd.DataFrame({
    'Feature': correlations.index,
    'Pearson Correlation': correlations.values,
    'Absolute Correlation': correlations.abs().values
})

# Sort features by absolute correlation (strength)
corr_df_sorted = corr_df.sort_values(by='Absolute Correlation', ascending=False)

# Optional: Select top k features
k = 5
top_features = corr_df_sorted.head(k)['Feature'].tolist()
X_selected = X[top_features]

# Output
print("Original shape:", X.shape)
print("Selected shape:", X_selected.shape)
print("\nTop Features Based on Pearson Correlation:\n", corr_df_sorted.head(k))


Original shape: (442, 10)
Selected shape: (442, 5)

Top Features Based on Pearson Correlation:
   Feature  Pearson Correlation  Absolute Correlation
2     bmi             0.586450              0.586450
8      s5             0.565883              0.565883
3      bp             0.441482              0.441482
7      s4             0.430453              0.430453
6      s3            -0.394789              0.394789


---
## 📈 **Spearman Correlation**
---

Spearman correlation measures the **monotonic relationship** between a numerical feature and a continuous or ordinal target using **ranked values**.

Value ranges from:  
- `+1`: Perfect increasing monotonic relationship  
- `-1`: Perfect decreasing monotonic relationship  
- `0`: No monotonic relationship  

| ✅ Use When                                        | ❌ Avoid When                          |
|--------------------------------------------------|----------------------------------------|
| Data has **non-linear but monotonic** patterns   | Data is **not monotonic**              |
| Features or target have **ranks or order**       | Features are **unordered categories**  |
| You want to **reduce outlier sensitivity**       | Relationship is **not monotonic**      |
| Both inputs are **numeric or ordinal**           |                                        |

In [25]:
import pandas as pd
from sklearn.datasets import load_diabetes
from scipy.stats import spearmanr

# Load dataset
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# Compute Spearman correlation for each feature with the target
correlations = X.apply(lambda col: spearmanr(col, y).correlation)

# Create a DataFrame for better readability
corr_df = pd.DataFrame({
    'Feature': correlations.index,
    'Spearman Correlation': correlations.values,
    'Absolute Correlation': correlations.abs().values
})

# Sort features by absolute correlation
corr_df_sorted = corr_df.sort_values(by='Absolute Correlation', ascending=False)

# Optional: Select top k features
k = 5
top_features = corr_df_sorted.head(k)['Feature'].tolist()
X_selected = X[top_features]

# Output
print("Original shape:", X.shape)
print("Selected shape:", X_selected.shape)
print("\nTop Features Based on Spearman Correlation:\n", corr_df_sorted.head(k))


Original shape: (442, 10)
Selected shape: (442, 5)

Top Features Based on Spearman Correlation:
   Feature  Spearman Correlation  Absolute Correlation
8      s5              0.589416              0.589416
2     bmi              0.561382              0.561382
7      s4              0.448931              0.448931
3      bp              0.416241              0.416241
6      s3             -0.410022              0.410022


---
## 📈 **Cramér's V**
---

Cramér's V measures the **association strength** between two **categorical variables**.  

It is based on the **Chi-squared statistic**, scaled between `0` (no association) and `1` (perfect association).

| ✅ Use When                                           | ❌ Avoid When                                |
|------------------------------------------------------|----------------------------------------------|
| Both **feature and target are categorical**          | Data is **numerical**                        |
| You want to measure **association, not correlation** | You need **directional/linear relationships** |
| Dataset is for **classification problems**           | Variables have **too many unique categories** |

In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

# 📊 Sample categorical dataset
data = {
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Green', 'Blue', 'Red'],
    'Size': ['S', 'M', 'L', 'S', 'M', 'L', 'S'],
    'Target': ['Buy', 'Don’t Buy', 'Buy', 'Buy', 'Don’t Buy', 'Buy', 'Buy']
}
df = pd.DataFrame(data)

# 🎯 Define categorical features and target
cat_features = ['Color', 'Size']
target_col = 'Target'

# Helper function to compute Cramér's V
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

# Compute Cramér's V for each feature
results = []
for col in cat_features:
    score = cramers_v(df[col], df[target_col])
    results.append((col, round(score, 4)))

# 🧾 Create DataFrame of scores
cramer_df = pd.DataFrame(results, columns=['Feature', "Cramér's V"])
cramer_df_sorted = cramer_df.sort_values(by="Cramér's V", ascending=False)

# Output
print("\nCategorical Features Ranked by Association with Target:\n")
print(cramer_df_sorted)



Categorical Features Ranked by Association with Target:

  Feature  Cramér's V
1    Size      1.0000
0   Color      0.5477


---
---
# **🧪 Statistical Tests**

| Method              | Target Type | Feature Type | Description                                        |
| ------------------- | ----------- | ------------ | -------------------------------------------------- |
| **✅ ANOVA F-Test**    | Categorical | Continuous   | Compares means of a feature across classes         |
| **✅ Chi-Square Test** | Categorical | Categorical  | Tests independence between feature and class label |
| **T-Test**          | Binary      | Continuous   | Like ANOVA, but for two classes only               |
| **Mann-Whitney U**  | Binary      | Continuous   | Non-parametric alternative to T-test               |


---
## 🧪 **ANOVA F-Test**
---

ANOVA F-test measures whether the **mean values of a numerical feature differ significantly across different categories of the target**.  

It helps identify features that best separate classes by testing variance between groups.

| ✅ Use When                                        | ❌ Avoid When                             |
|---------------------------------------------------|------------------------------------------|
| Features are **numerical**                         | Features or target are **categorical** (for features) |
| Target is **categorical** (classification task)   | Target is **continuous** (regression)   |
| Want to find features with **significant group differences** | Feature distributions are **non-normal** or have **unequal variances** |

In [27]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest, f_classif

# Load dataset (features are numerical, target categorical)
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Apply ANOVA F-test to select top k features
k = 2
anova_selector = SelectKBest(score_func=f_classif, k=k)
X_selected = anova_selector.fit_transform(X, y)

# Get mask of selected features and F-scores
mask = anova_selector.get_support()
scores = anova_selector.scores_

# Create DataFrame for better visualization
feature_scores_df = pd.DataFrame({
    'Feature': data.feature_names,
    'ANOVA F-Score': scores,
    'Selected': mask
}).sort_values(by='ANOVA F-Score', ascending=False)

# Output
print("Original shape:", X.shape)
print("Selected shape:", X_selected.shape)
print("\nFeature Scores:\n", feature_scores_df)


Original shape: (150, 4)
Selected shape: (150, 2)

Feature Scores:
              Feature  ANOVA F-Score  Selected
2  petal length (cm)    1180.161182      True
3   petal width (cm)     960.007147      True
0  sepal length (cm)     119.264502     False
1   sepal width (cm)      49.160040     False


---
## 🧪 **Chi-Square Test**
---

Chi-Square test measures the **dependence between categorical features and a categorical target** by testing if their distributions are independent.  

It helps select features that are strongly associated with the target.

| ✅ Use When                                      | ❌ Avoid When                             |
|-------------------------------------------------|------------------------------------------|
| Features are **categorical**                     | Features are **numerical** without binning |
| Target is **categorical**                        | Target is **continuous**                   |
| Want to find features **associated with target** | Small sample size (expected counts < 5) |


In [28]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import KBinsDiscretizer

# Load dataset (features numeric, target categorical)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Chi-Square requires non-negative discrete features, so discretize numeric features
discretizer = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='uniform')
X_binned = discretizer.fit_transform(X)

# Apply Chi-Square test to select top k features
k = 5
chi2_selector = SelectKBest(score_func=chi2, k=k)
X_selected = chi2_selector.fit_transform(X_binned, y)

# Get mask and scores
mask = chi2_selector.get_support()
scores = chi2_selector.scores_

# Create DataFrame for better visualization
feature_scores_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Chi-Square Score': scores,
    'Selected': mask
}).sort_values(by='Chi-Square Score', ascending=False)

# Output
print("Original shape:", X.shape)
print("Selected shape:", X_selected.shape)
print("\nFeature Scores:\n", feature_scores_df)


Original shape: (569, 30)
Selected shape: (569, 5)

Feature Scores:
                     Feature  Chi-Square Score  Selected
7       mean concave points        638.003011      True
6            mean concavity        567.531043      True
27     worst concave points        541.792008      True
23               worst area        499.230051      True
22          worst perimeter        417.010141      True
20             worst radius        414.588612     False
26          worst concavity        396.942853     False
3                 mean area        356.886992     False
10             radius error        303.568899     False
2            mean perimeter        302.579971     False
12          perimeter error        294.353872     False
0               mean radius        285.808388     False
25        worst compactness        264.922050     False
13               area error        255.673825     False
5          mean compactness        252.898914     False
21            worst texture        

---
---
# **📙 Information-Based Methods Tests**

| Method                      | Target Type               | Feature Type | Description                                                                 |
| --------------------------- | ------------------------- | ------------ | --------------------------------------------------------------------------- |
| **Mutual Information (MI)** | Categorical or Continuous | Any          | Measures how much knowing one variable reduces uncertainty of the other     |
| **Information Gain (IG)**   | Categorical               | Any          | Derived from entropy — how much information a feature gives about the class |
| **Gain Ratio**              | Categorical               | Any          | Normalized version of Information Gain                                      |
| **Symmetrical Uncertainty** | Categorical               | Any          | MI normalized between 0 and 1                                               |


---
## 📙 **Mutual Information**
---

Mutual Information measures the **dependency between features and the target**, capturing **any kind of relationship** (linear or non-linear).  

Higher MI means more shared information between the feature and target.

| ✅ Use When                                  | ❌ Avoid When                         |
|---------------------------------------------|-------------------------------------|
| Features are **numerical or categorical**   | Very small datasets (MI estimates can be unstable) |
| Target is **categorical** (classification) |                                     |
| Target is **continuous** (regression)       |                                     |
| Want a **non-linear, model-agnostic filter** |                                     |

In [29]:
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

# Example for classification
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Compute Mutual Information for classification task
mi_scores = mutual_info_classif(X, y, discrete_features=False, random_state=42)

mi_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Mutual Information Score': mi_scores
}).sort_values(by='Mutual Information Score', ascending=False)

print("Classification - Mutual Information Scores:\n", mi_df)


# Example for regression
data_reg = load_diabetes()
X_reg = pd.DataFrame(data_reg.data, columns=data_reg.feature_names)
y_reg = data_reg.target

# Compute Mutual Information for regression task
mi_scores_reg = mutual_info_regression(X_reg, y_reg, discrete_features=False, random_state=42)

mi_df_reg = pd.DataFrame({
    'Feature': data_reg.feature_names,
    'Mutual Information Score': mi_scores_reg
}).sort_values(by='Mutual Information Score', ascending=False)

print("\nRegression - Mutual Information Scores:\n", mi_df_reg)


Classification - Mutual Information Scores:
                     Feature  Mutual Information Score
22          worst perimeter                  0.471842
23               worst area                  0.464313
20             worst radius                  0.451230
7       mean concave points                  0.438806
27     worst concave points                  0.436255
2            mean perimeter                  0.402361
6            mean concavity                  0.375447
0               mean radius                  0.362276
3                 mean area                  0.360023
13               area error                  0.340759
26          worst concavity                  0.315259
12          perimeter error                  0.275614
10             radius error                  0.249301
25        worst compactness                  0.225211
5          mean compactness                  0.213439
17     concave points error                  0.125415
21            worst texture          

---
---
# **🎲 Variance-Based Methods Tests**

| Method                            | Target Type         | Feature Type | Description                                                                       |
| --------------------------------- | ------------------- | ------------ | --------------------------------------------------------------------------------- |
| **Variance Threshold**            | None (unsupervised) | Continuous   | Removes features with low variance (assumes low variance = low information)       |
| **Coefficient of Variation (CV)** | None                | Continuous   | Ratio of std dev to mean — filters features with too much or too little variation |



---
## 🎲 **Variance Threshold**
---

**Variance Threshold** removes features with **low variance**, assuming they provide **little to no information**.  

It is a **baseline filter** that doesn’t consider the target variable.

| ✅ Use When                                     | ❌ Avoid When                              |
|------------------------------------------------|--------------------------------------------|
| Features are **numerical**                     | Target information is important            |
| You want to remove **constant or near-constant** features | Features are **categorical** (needs encoding first) |
| Doing **initial preprocessing** to clean data  | You expect even low-variance features to matter |

In [30]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import VarianceThreshold

# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)

# Initialize VarianceThreshold - default threshold = 0 (removes constant features)
selector = VarianceThreshold(threshold=0.01)
X_reduced = selector.fit_transform(X)

# Get selected feature names
mask = selector.get_support()
selected_features = X.columns[mask]

# Create DataFrame showing variance for each feature
variances = X.var()
var_df = pd.DataFrame({
    'Feature': X.columns,
    'Variance': variances,
    'Selected': mask
}).sort_values(by='Variance', ascending=False)

# Output
print("Original shape:", X.shape)
print("Selected shape:", X_reduced.shape)
print("\nVariance Summary:\n", var_df)


Original shape: (569, 30)
Selected shape: (569, 14)

Variance Summary:
                                          Feature       Variance  Selected
worst area                            worst area  324167.385102      True
mean area                              mean area  123843.554318      True
area error                            area error    2069.431583      True
worst perimeter                  worst perimeter    1129.130847      True
mean perimeter                    mean perimeter     590.440480      True
worst texture                      worst texture      37.776483      True
worst radius                        worst radius      23.360224      True
mean texture                        mean texture      18.498909      True
mean radius                          mean radius      12.418920      True
perimeter error                  perimeter error       4.087896      True
texture error                      texture error       0.304316      True
radius error                        radi

---
---
# **🧩 Feature Discretization Methods**

| Method               | Target Type | Feature Type | Description                                                                    |
| -------------------- | ----------- | ------------ | ------------------------------------------------------------------------------ |
| **Relief / ReliefF** | Categorical | Any          | Weights features by how well they separate near instances of different classes |
| **Fisher Score**     | Categorical | Continuous   | Ratio of inter-class separation to intra-class variance                        |




---
---
# **🧠 Other Useful Methods**

| Method                                         | Target Type | Feature Type | Description                                                                             |
| ---------------------------------------------- | ----------- | ------------ | --------------------------------------------------------------------------------------- |
| **Univariate Feature Selection (SelectKBest)** | Any         | Any          | Wrapper for applying above tests (e.g., ANOVA, Chi2, MI)                                |
| **SelectPercentile**                           | Any         | Any          | Same as SelectKBest, but selects top X%                                                 |
| **Maximal Information Coefficient (MIC)**      | Continuous  | Any          | Captures linear & non-linear associations (via [minepy](https://minepy.readthedocs.io)) |
| **Gini Index / Gini Impurity**                 | Categorical | Any          | Used in tree models — can be adapted as a filter                                        |


---
## 🧠 **Linear Discriminant Analysis (LDA)**
---

Linear Discriminant Analysis is a supervised dimensionality reduction technique that transforms features into a space that maximizes class separability by leveraging class labels.

| ✅ Use When                                            | ❌ Avoid When                                 |
| ----------------------------------------------------- | -------------------------------------------- |
| Performing **classification** (target is categorical) | Task is **regression** (continuous target)   |
| Need to reduce dimensionality with **class labels**   | Classes are **not well separated**           |
| Want to improve **class separability**                | Data does **not follow normal distribution** |
| You expect **linear boundaries** between classes      | Features are **non-linearly separable**      |


In [31]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Load classification dataset
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Initialize and fit LDA
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X, y)

# Convert to DataFrame
X_lda_df = pd.DataFrame(X_lda, columns=[f'LDA{i+1}' for i in range(X_lda.shape[1])])

# Output
print("Original shape:", X.shape)
print("LDA reduced shape:", X_lda_df.shape)
print(X_lda_df.head())


Original shape: (178, 13)
LDA reduced shape: (178, 2)
       LDA1      LDA2
0 -4.700244  1.979138
1 -4.301958  1.170413
2 -3.420720  1.429101
3 -4.205754  4.002871
4 -1.509982  0.451224


---
---
---